# DizerCore LoRA Training — auto-matches installed model

Train a TrinityCore-specific LoRA adapter on Kaggle's free T4/P100 GPU.

**Setup:**
1. Create a Kaggle dataset and upload `dizercore-dataset.jsonl` (one file — it carries the base-model tag)
2. Attach it to this notebook (Add Data), then fix the DATASET path below to match your dataset slug
3. Settings → Accelerator → T4 GPU
4. Run All

Runtime: ~2 hours on T4 for 1.5B. 3B is slower.

In [ ]:
import subprocess, sys, os, gc, json

print('=== Install ===', flush=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'transformers', 'peft', 'bitsandbytes',
                'accelerate', 'datasets', 'hf_transfer'], check=True)

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# ---------- Config ----------
# Point DATASET at wherever your Kaggle dataset mounts the JSONL.
DATASET = '/kaggle/input/datasets/vekzla/dizercore-wow/dizercore-dataset.jsonl'
WORK = '/kaggle/working/dizercore'
ADAPTER = f'{WORK}/adapter'
MERGED = f'{WORK}/merged'
GGUF_F16 = f'{WORK}/dizercore-f16.gguf'
GGUF_Q4 = f'{WORK}/dizercore-q4_k_m.gguf'
os.makedirs(WORK, exist_ok=True)

assert os.path.isfile(DATASET), f'Dataset not found at {DATASET} — check your Kaggle dataset slug'

# Base model comes from the dataset itself (line 1 meta record written
# by dataset-builder.py on the Pi). Falls back to 1.5B-Instruct.
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
try:
    with open(DATASET) as _f:
        BASE = json.loads(_f.readline()).get('base_model', BASE)
except Exception as _e:
    print(f'No base_model tag in dataset ({_e}); using default')
print('Base model:', BASE)
print('Dataset:', DATASET)

In [ ]:
print('=== Load dataset ===', flush=True)
from datasets import load_dataset

ds = load_dataset('json', data_files=DATASET)['train']
ds = ds.filter(lambda r: not r.get('_meta', False))
print(f'{len(ds)} training examples (meta row excluded)')

def fmt(ex):
    instr = ex['instruction']
    inp = ex.get('input', '') or ''
    out = ex['output']
    if inp:
        instr = instr + '\n\n' + inp
    return {
        'text': (
            '<|im_start|>system\nYou are a TrinityCore development assistant.<|im_end|>\n'
            f'<|im_start|>user\n{instr}<|im_end|>\n'
            f'<|im_start|>assistant\n{out}<|im_end|>'
        )
    }

ds = ds.map(fmt, remove_columns=ds.column_names)
print(ds[0]['text'][:400])

In [ ]:
print('=== Load base model (4-bit) ===', flush=True)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb,
                                             device_map='auto')
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                  'gate_proj', 'up_proj', 'down_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
print('=== Train ===', flush=True)
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

def tokenize(ex):
    return tokenizer(ex['text'], truncation=True, max_length=1024, padding=False)

tok = ds.map(tokenize, remove_columns=ds.column_names)

args = TrainingArguments(
    output_dir=f'{WORK}/checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=25,
    save_total_limit=1,
    save_only_model=True,
    fp16=True,
    gradient_checkpointing=True,
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=tok,
                  data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
trainer.train()

model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print('Adapter saved:', ADAPTER, flush=True)

In [ ]:
print('=== Merge LoRA into base ===', flush=True)
del model, trainer
gc.collect()
torch.cuda.empty_cache()

from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16,
                                            device_map='auto')
merged = PeftModel.from_pretrained(base, ADAPTER)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED)
tokenizer.save_pretrained(MERGED)
del base, merged
gc.collect()
torch.cuda.empty_cache()
print('Merged model saved:', MERGED, flush=True)

In [ ]:
print('=== Build llama.cpp ===', flush=True)
LLAMA_DIR = '/kaggle/working/llama.cpp'
if not os.path.isdir(LLAMA_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggml-org/llama.cpp', LLAMA_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                f'{LLAMA_DIR}/requirements.txt'], check=True)
subprocess.run(['cmake', '-B', f'{LLAMA_DIR}/build'],
               cwd=LLAMA_DIR, check=True)
subprocess.run(['cmake', '--build', f'{LLAMA_DIR}/build',
                '--target', 'llama-quantize', '-j', '4'], check=True)

print('=== Convert to GGUF (f16) ===', flush=True)
subprocess.run([sys.executable, f'{LLAMA_DIR}/convert_hf_to_gguf.py',
                MERGED, '--outfile', GGUF_F16, '--outtype', 'f16'], check=True)

print('=== Quantizing to Q4_K_M ===', flush=True)
subprocess.run([f'{LLAMA_DIR}/build/bin/llama-quantize',
                GGUF_F16, GGUF_Q4, 'Q4_K_M'], check=True)

size_mb = os.path.getsize(GGUF_Q4) / 1024 / 1024
print('=== DONE ===', flush=True)
print(f'Base trained: {BASE}', flush=True)
print(f'GGUF: {GGUF_Q4} ({size_mb:.0f} MB)', flush=True)
print('Download it from the Output panel, then upload via the Web UI Training tab.', flush=True)

In [ ]:
# ============ DOWNLOAD ============
# On Kaggle, the file is in the Output panel (right sidebar) under
# dizercore/dizercore-q4_k_m.gguf — FileLink doesn't render there.
# In Colab/local Jupyter this cell gives you a clickable download link.
import os
if os.path.isdir('/kaggle'):
    print('On Kaggle: open the Output panel and download:')
    print(' ', GGUF_Q4)
else:
    from IPython.display import FileLink, display
    print('Click to download:')
    display(FileLink(GGUF_Q4, result_html_prefix='Download: '))